# Threshold tools

This notebook demonstrates how to use `climakitae` to evaluate threshold exceedences for extreme weather events in downscaled climate model simulations. 

`climakitae` uses the Annual Maxima Series (AMS) method, as opposed to the alternative Peaks-Over-Threshold (POT) method, for extreme value analysis. In the AMS method, the maximum (or minimum) variable value is obtained for each year. A statistical distribution is fitted to the values and can then be used to obtain return probabilities, values, or periods. Users will have to option to choose their distribution along with other customizations.

**Terms used in this notebook**:
- __Return probabilities__: The probability of a threshold being exceeded within a given period of time. 
    - For example, the maximum temperature at a location has a 10% chance of exceeding 105°F in any year.
- __Return values__: A threshold expected to be exceeded at least once in a set period of time. 
    - A maximum temperature value that has a 10% annual return probability is the 10-year maximum temperature return value.
- __Return periods__: The frequency with which a threshold is expected to be exceeded at least once.
    - If the return probability of exceeding 150 mm of precipitation is 10%, the return period of that event is 10 years.

**Intended Application**: As a user, I want to learn how to:
- **<span style="color:#FF0000">Generate return probabilities, return values, and return periods for climate simulations with climakitae.</span>**
- **<span style="color:#FF0000">Visualize the spatial distribution of return value results across a region.</span>**

**Runtime**: With the default settings, this notebook takes approximately **5 minutes** to run from start to finish. Modifications to selections may increase the runtime.

**References**: The techniques in this notebook come from applications of extreme value theory to climate data. For further reading on this topic, see [Cooley 2009](https://link.springer.com/article/10.1007/s10584-009-9627-x).


### 1-in-X options

The following arguments can be passed to `one_in_x` in the `metric_calc` processor:

| Argument | Options | Required? | Notes |
|---|---|---|---|
| `return_periods` | List of ints | Required (or `return_values`) | Return periods in years, e.g. `[10, 25, 50]` |
| `return_values` | List of floats | Required (or `return_periods`) | Threshold values, e.g. `[105]` |
| `extremes_type` | `"max"`, `"min"` | Optional | Default: `"max"` |
| `distribution` | `"gev"`, `"genpareto"`, `"gumbel"`, `"weibull"`, `"pearson3"`, `"gamma"` | Optional | Default: `"gev"` |
| `event_duration` | `(int, "day")` | Optional | Window for sub-daily events |
| `grouped_duration` | `(int, "day")` | Optional | Consecutive-day event window for multi-day extremes |
| `block_size` | int (years) | Optional | Default: 1 |
| `goodness_of_fit_test` | bool | Optional | Default: `True`; runs KS test, returns p-values |

In [ ]:
import climakitae as ck
from climakitae.util.colormap import read_ae_colormap

import matplotlib.pyplot as plt

## 1. Return values: precipitation at a single location

In this example, we calculate 1-in-X return values for daily precipitation at a single station (KSAC) across two warming levels.

In [ ]:
# Initialize the interface
cd = ck.ClimateData(verbosity=0)

one_in_x_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("prec")
    .processes(
        {
            "warming_level": {"warming_levels": [0.8, 2.0]},
            "convert_units": "inches/d",
            "clip": "KSAC",
            "metric_calc": {
                "one_in_x": {
                    "return_periods": [10, 25, 50],
                    "extremes_type": "max",
                }
            },
        }
    )
    .get()
)

In [ ]:
# Removing `EC-Earth3-Veg` because it has incomplete WL 0.8
sims = [s.item() for s in one_in_x_data.sim if "EC-Earth3-Veg" not in s.item()]
one_in_x_data = one_in_x_data.sel(sim=sims)

The p-value from the Kolmogorov-Smirnov (KS) test measures how well the chosen distribution fits the annual maxima series. A p-value above 0.05 indicates an acceptable fit at the 95% confidence level. Low p-values suggest the distribution may not be appropriate for the data-- try a different `distribution` option in that case.

In [ ]:
# Examining p-values across simulation, warming_level, and one_in_x combinations
one_in_x_data.to_dataframe()[["p_values"]]

In [ ]:
# Looking at the `return_values` now that we've confirmed the p-values are all reasonable
one_in_x_data.to_dataframe()[["return_values"]].groupby(
    ["warming_level", "one_in_x"]
).mean()

**As expected, longer return periods correspond to higher return values, reflecting the fact that more severe events occur less frequently.**
<br>
<br>

## 2. Return periods: temperature over a region

In this example, we used a time-based approach to find the return period for a specific temperature threshold (105°F for a 3-day heat wave) across San Bernardino County for simulation years 2040-2060.  We then display the results as a map. 

This example demonstrates how to use the `event_duration` and `grouped_duration` keywords together to compute return periods for multi-day events, in this case a three-day event based on daily maximum temperature. 

In [ ]:
# Set the three-day heatwave temperature
tmax_3day = 115

# Initialize the interface
cd = ck.ClimateData(verbosity=0)

one_in_x_temp = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("t2max")
    .processes(
        {
            "time_slice": (2040, 2060),
            "convert_units": "degF",
            "clip": "San Bernardino County",
            "metric_calc": {
                "one_in_x": {
                    "return_values": [tmax_3day],
                    "block_size": 1,
                    "extremes_type": "max",
                    "event_duration": (1, "day"),
                    "grouped_duration": (3, "day"),  # select consecutive 3-day events
                }
            },
        }
    )
    .get()
)

### Visualize the return periods
Next we take the median of the return period in the five simulations and create a map.  

**Larger return periods indicate less frequent events (i.e. a longer wait time between events).** These areas have a lighter color on the map.   
**Areas with smaller return periods are more susceptible to 3-day heat waves at the selected temperature.** These are the darker colors on the map. Areas with a return period of 1 have a 100% probability of experiencing the event each year based on this analysis.

In [ ]:
fig, ax = plt.subplots()
one_in_x_temp["return_periods"].median("sim").sel(one_in_x=tmax_3day).plot(
    vmax=40,
    vmin=1,
    cmap=read_ae_colormap("ae_orange").reversed(),
    x="lon",
    y="lat",
    ax=ax,
    cbar_kwargs={"label": "Return Period (years)"},
)
ax.set_title(
    f"Return period of a 3-day, {tmax_3day}°F heat wave\nSimulation median for 2040-2060"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 3. Different return periods across WLs for a given threshold

This example considers **how the frequency of a 1-in-X event changes across warming levels.** 

Here, we take the median 10-year precipitation return value at WL 0.8 from Section 1, and find its return period at WL 2.0.

In [ ]:
# Extract the median 10-year return value at WL 0.8 as our threshold
threshold_val = float(
    one_in_x_data.to_dataframe()["return_values"]
    .xs(0.8, level="warming_level")
    .xs(10, level="one_in_x")
    .median()
)
print(f"10-year return value at WL 0.8: {threshold_val:.2f} inches/day")

In [ ]:
# Find the return period for that threshold at both warming levels
cd = ck.ClimateData(verbosity=0)

return_period_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("prec")
    .processes(
        {
            "warming_level": {"warming_levels": [0.8, 2.0]},
            "convert_units": "inches/d",
            "clip": "KSAC",
            "metric_calc": {
                "one_in_x": {
                    "return_values": [threshold_val],
                    "extremes_type": "max",
                }
            },
        }
    )
    .get()
)

# Dropping simulations with incomplete WL data
sims = [s.item() for s in return_period_data.sim if "EC-Earth3-Veg" not in s.item()]
return_period_data = return_period_data.sel(sim=sims)

In [ ]:
# Plotting the return period for the same return value across WL 0.8 and 2.0
rp_summary = (
    return_period_data.to_dataframe()[["return_periods"]]
    .groupby("warming_level")
    .median()
)

fig, ax = plt.subplots()
rp_summary["return_periods"].plot(kind="bar", ax=ax, color=["steelblue", "tomato"])
ax.set_xlabel("Warming Level (°C)")
ax.set_ylabel("Return Period (years)")
ax.set_title(f"Return period of {threshold_val:.2f} in/day precipitation event")
ax.set_xticklabels([f"WL {wl}" for wl in rp_summary.index], rotation=0)
ax.set_ylim(6, 12)
plt.tight_layout()
plt.show()

**At WL 0.8, this event is by construction a 10-year return event. A shorter return period at WL 2.0 means the same precipitation amount is expected more frequently, which indicates the intensification of extreme precipitation at a higher warming level.**
<br>
<br>

## 4. Other notebooks that use threshold analyses
Now that you've seen how the different `1-in-X` calculations can be performed using `climakitae`, here's a few other notebooks that we support that use similar functionality for various other use cases:

- **[Vulnerability Assessment](../collaborative/IOU/vulnerability_assessment/vulnerability_assessment.ipynb)**: Pre-built metric tables for IOU utility planning. Uses `cava_data()` to generate standardized 1-in-X outputs, percentiles,  across multiple variables.

- **[Event Finder](../work-in-progress/event_finder.ipynb)**: Identifies specific historical or projected extreme events that match a given threshold or 1-in-X event, useful for understanding specific behavior.

- **[Custom 1-in-X](../work-in-progress/custom_1_in_x.ipynb)**: Calculates 1-in-X return values for customized metrics and/or variables.

- **[1-in-X into 8760](../collaborative/IOU/8760/one_in_x_in_8760.ipynb)**: Generates an 8760 timeseries, and inserts 1-in-X events into them, to capture what yearly behavior would look like with an extreme event.